In [1]:
import pandas as pd
import numpy as np

def build_granular_history(years=[2024, 2025, 2026]):
    pd.options.display.float_format = '{:,.0f}'.format
    yearly_dataframes = []
    
    for year in years:
        # 1. Load Data
        df_needs = pd.read_csv(f"../../data/hpc_hno_{year}.csv", low_memory=False)
        df_needs = df_needs[1:] 
        
        # 2. Clean Numbers
        df_needs["In Need"] = df_needs["In Need"].astype(str).str.replace(',', '')
        df_needs["In Need"] = pd.to_numeric(df_needs["In Need"], errors='coerce').fillna(0)
        
        # 3. GRANULAR FILTERING
        # Keep Admin 1 (Provinces). Exclude Admin 2 (Districts) to avoid double counting.
        df_regional = df_needs[
            (df_needs['Admin 1 PCode'].notna()) & 
            (df_needs['Admin 2 PCode'].isna())
        ].copy()
        
        # Keep the demographic Category. If it's blank, label it 'Total' for that region.
        df_regional['Category'] = df_regional['Category'].fillna('Total')
        df_regional['Year'] = year
        
        yearly_dataframes.append(df_regional)
        
    # Stack all years together
    df_history = pd.concat(yearly_dataframes, ignore_index=True)
    
    # 4. Pivot Clusters
    # We expand our index to include the region, demographic category, and year
    df_pivot = pd.pivot_table(
        df_history, 
        values='In Need', 
        index=['Country ISO3', 'Admin 1 PCode', 'Admin 1 Name', 'Category', 'Year'], 
        columns='Cluster', 
        aggfunc='sum', 
        fill_value=0
    ).reset_index()
    
    # Rename cluster columns
    idx_cols = ['Country ISO3', 'Admin 1 PCode', 'Admin 1 Name', 'Category', 'Year']
    cluster_cols = [col for col in df_pivot.columns if col not in idx_cols]
    df_pivot = df_pivot.rename(columns={col: f"In Need - {col}" for col in cluster_cols})
    
    return df_pivot

# Generate the historical dataset
df_regional_history = build_granular_history([2024, 2025, 2026])

KeyError: 'Admin 1 PCode'

In [ ]:
# 1. Load the Admin 1 (Regional) Population Data
df_pop_admin1 = pd.read_csv("../../data/cod_population_admin1.csv", low_memory=False)
df_pop_admin1["Population"] = df_pop_admin1["Population"].astype(str).str.replace(',', '')
df_pop_admin1["Population"] = pd.to_numeric(df_pop_admin1["Population"], errors='coerce').fillna(0)

# Extract the overall 'Total' population for each region (avoiding age/gender double counts in COD)
df_pop_reg = df_pop_admin1[
    df_pop_admin1['Gender'].str.lower().isin(['all', 'total', 'both']) | 
    df_pop_admin1['Gender'].isna()
]
df_pop_reg = df_pop_reg.groupby('ADM1_PCODE')['Population'].max().reset_index()
df_pop_reg = df_pop_reg.rename(columns={'ADM1_PCODE': 'Admin 1 PCode', 'Population': 'Regional Population'})

# 2. Merge Population into our Historical Data
# We use a Left Join so we don't lose HNO data just because a population figure is missing
df_master_granular = pd.merge(df_regional_history, df_pop_reg, on='Admin 1 PCode', how='left')

# 3. Dynamically Calculate Relative Needs for all Clusters
in_need_cols = [col for col in df_master_granular.columns if col.startswith('In Need - ')]

for col in in_need_cols:
    sector_name = col.replace('In Need - ', '')
    pct_col_name = f"% Pop In Need - {sector_name}"
    
    df_master_granular[pct_col_name] = (df_master_granular[col] / df_master_granular['Regional Population']) * 100
    df_master_granular[pct_col_name] = df_master_granular[pct_col_name].replace([np.inf, -np.inf], np.nan).fillna(0)

# Check a specific region over time (e.g., a province in Afghanistan)
afg_sample = df_master_granular[df_master_granular['Country ISO3'] == 'AFG']
afg_sample.head(10)